# Lab 15 — LangGraph plan-and-execute bridge (reference solution)

Canonical LangGraph rebuild of [Lab 12's plan-and-execute pattern](../../12-plan-and-execute-from-scratch/solution/README.md). The strong framework value case: `Send` replaces ~70 lines of `ThreadPoolExecutor` + `threading.Lock` with ~10 lines.

> 📖 See [`solution/README.md`](./README.md) for the design choices flagged.
> ⏱ Run time: 20-40 seconds end-to-end.


## Setup

Provider-agnostic chat model. Same pattern as Lab 14.

In [ ]:
import json
import os
import re
import warnings
from typing import Annotated, Any, Literal, TypedDict

from dotenv import load_dotenv

import pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

PROVIDER = os.environ.get("PROVIDER", "openai")
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]

if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=MODEL, temperature=0)
else:
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model=MODEL, temperature=0)

print(f"Provider: {PROVIDER}, model: {MODEL}")


## Plan schemas + `validate_graph`

Carried verbatim from Lab 12: `StrictModel` with `extra="forbid"`, Kahn's-algorithm cycle detection, parallel-group violation check, unknown-tool check.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class PlanStep(StrictModel):
    id: str = Field(description="Unique step ID, e.g. 'step_1'")
    description: str = Field(description="What this step does, self-contained.")
    tool: str = Field(description="Tool to invoke. Must be in executor registry.")
    args: dict = Field(default_factory=dict, description="Arguments to the tool.")
    depends_on: list[str] = Field(default_factory=list)
    parallel_group: str | None = Field(default=None)


class Plan(StrictModel):
    steps: list[PlanStep] = Field(description="The ordered list of steps.")


MAX_PLAN_STEPS = 8
EXECUTOR_MAX_STEPS = 4
MAX_REPLANS = 2


def validate_graph(plan: Plan, available_tools: set[str]) -> list[str]:
    """Return list of validation errors (empty list = valid plan)."""
    errors: list[str] = []
    step_ids = {s.id for s in plan.steps}
    if len(step_ids) != len(plan.steps):
        errors.append("duplicate step IDs")
    for s in plan.steps:
        if s.tool not in available_tools:
            errors.append(f"step {s.id} uses unknown tool '{s.tool}'")
        for dep in s.depends_on:
            if dep not in step_ids:
                errors.append(f"step {s.id} depends on unknown step '{dep}'")
            if dep == s.id:
                errors.append(f"step {s.id} depends on itself")
    by_group: dict[str, list[PlanStep]] = {}
    for s in plan.steps:
        if s.parallel_group is not None:
            by_group.setdefault(s.parallel_group, []).append(s)
    for group, members in by_group.items():
        member_ids = {s.id for s in members}
        for s in members:
            for dep in s.depends_on:
                if dep in member_ids:
                    errors.append(
                        f"parallel_group '{group}' has step {s.id} depending on group-mate {dep}"
                    )
    # Kahn's algorithm for cycle detection
    incoming = {s.id: set(s.depends_on) for s in plan.steps}
    no_in = [sid for sid, deps in incoming.items() if not deps]
    visited: list[str] = []
    while no_in:
        n = no_in.pop()
        visited.append(n)
        for sid, deps in incoming.items():
            if n in deps:
                deps.discard(n)
                if not deps and sid not in visited and sid not in no_in:
                    no_in.append(sid)
    if len(visited) < len(plan.steps):
        errors.append("cycle detected")
    return errors


## State schema with reducer

The `_merge_results` reducer on `completed` is the framework primitive that makes parallel `Send` dispatch work. Without it, parallel executor updates clobber each other.

In [ ]:
def _merge_results(existing: dict, update: dict) -> dict:
    """Reducer: merge per-step results from parallel Send dispatches."""
    merged = dict(existing or {})
    merged.update(update or {})
    return merged


class PlanState(TypedDict):
    task: str
    plan: list[dict]                                # serialized PlanStep list
    completed: Annotated[dict, _merge_results]      # step_id → result dict
    failed: list[dict]
    replan_count: int
    final_answer: str
    status: str                                     # "ok", "partial_after_cap", "error"


## Web tools

`web_search` + `fetch_page` carry over from Lab 10/12/14. Same envelope shape.

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

from langchain_core.tools import tool

warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

_RECENCY_MAP = {"any": None, "day": "d", "week": "w", "month": "m", "year": "y"}
USER_AGENT = ("AgenticAIEngineer-CourseLab/0.1 "
              "(https://github.com/MHHamdan/Agentic-AI-Engineer)")


@tool
def web_search(query: str, recency: str = "any", max_results: int = 8) -> str:
    """Search the web. Returns up to max_results items with title, url, snippet."""
    if not query or not query.strip():
        return json.dumps({"status": "error", "kind": "other", "detail": "empty query"})
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(query=query.strip(), region="us-en", safesearch="moderate",
                             timelimit=_RECENCY_MAP.get(recency),
                             max_results=max_results, backend="auto")
    except (RatelimitException, TimeoutException, DDGSException) as e:
        kind = {"RatelimitException": "rate_limit",
                "TimeoutException": "timeout"}.get(type(e).__name__, "other")
        return json.dumps({"status": "error", "kind": kind, "detail": str(e)})
    except Exception as e:
        return json.dumps({"status": "error", "kind": "other",
                           "detail": f"{type(e).__name__}: {e}"})
    if not raw:
        return json.dumps({"status": "empty", "query": query})
    return json.dumps({"status": "ok",
        "results": [{"title": (r.get("title") or "").strip(),
                     "url": (r.get("href") or "").strip(),
                     "snippet": (r.get("body") or "").strip()}
                    for r in raw if r.get("href")][:max_results]})


@tool
def fetch_page(url: str, max_chars: int = 8000) -> str:
    """Fetch the full text content of a URL."""
    if not url or not url.startswith(("http://", "https://")):
        return json.dumps({"status": "error", "url": url, "kind": "other",
                           "detail": "invalid url"})
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=15)
        resp.raise_for_status()
    except requests.HTTPError:
        kind = "http_5xx" if 500 <= resp.status_code < 600 else "http_4xx"
        return json.dumps({"status": "error", "url": url, "kind": kind,
                           "detail": f"HTTP {resp.status_code}"})
    except requests.RequestException as e:
        return json.dumps({"status": "error", "url": url, "kind": "other",
                           "detail": f"{type(e).__name__}: {e}"})
    soup = BeautifulSoup(resp.text, "html.parser")
    for el in soup(["script", "style", "nav", "footer", "aside"]):
        el.decompose()
    text = re.sub(r"\s+", " ", soup.get_text(separator=" ")).strip()[:max_chars]
    title = (soup.title.string or "").strip() if soup.title else ""
    return json.dumps({"status": "ok", "url": url, "title": title, "text": text})


EXECUTOR_TOOLS = [web_search, fetch_page]

EXECUTOR_TOOL_REGISTRY = {
    "web_search": {
        "description": "Search the web. Returns title/url/snippet results.",
        "args_schema": {"query": "string, 3-8 specific words",
                        "recency": "one of: any, day, week, month, year",
                        "max_results": "integer 1-10"},
    },
    "fetch_page": {
        "description": "Fetch the full text content of a single URL.",
        "args_schema": {"url": "string", "max_chars": "integer"},
    },
}


## Planner node

Validation-then-retry loop in-node. Emits a `Plan` as JSON; rejects malformed; retries with validation feedback.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage


def _format_tool_registry(reg: dict) -> str:
    lines = []
    for name, info in reg.items():
        lines.append(f"- {name}: {info['description']}")
        for arg, desc in info["args_schema"].items():
            lines.append(f"    args.{arg}: {desc}")
    return "\n".join(lines)


PLANNER_SYSTEM_PROMPT_TEMPLATE = """You are a planner. Given a user task, emit a Plan as JSON:

{{
  "steps": [
    {{"id": "step_1", "description": "...", "tool": "...", "args": {{...}},
      "depends_on": [...], "parallel_group": "..." or null}},
    ...
  ]
}}

EXECUTOR has these tools (and ONLY these):
{tool_registry}

RULES:
1. Atomic steps (one tool call per step).
2. Explicit depends_on.
3. parallel_group members must not depend on each other.
4. Self-contained descriptions.
5. At most {max_steps} steps.

Return ONLY JSON. No prose. No markdown fences.
"""


def _strip_fences(raw: str) -> str:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if "\n" in raw:
            raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()
    return raw


def planner_node(state: PlanState) -> dict:
    """Emit a Plan, validate, retry on failure (up to 2 retries)."""
    sys_prompt = PLANNER_SYSTEM_PROMPT_TEMPLATE.format(
        tool_registry=_format_tool_registry(EXECUTOR_TOOL_REGISTRY),
        max_steps=MAX_PLAN_STEPS,
    )
    task = state["task"]
    feedback = ""
    last_error = ""
    for _attempt in range(3):
        prompt = f"USER TASK: {task}\n{feedback}"
        resp = llm.invoke([SystemMessage(content=sys_prompt),
                            HumanMessage(content=prompt)])
        raw = _strip_fences(resp.content if isinstance(resp.content, str) else str(resp.content))
        try:
            parsed = json.loads(raw)
            plan = Plan.model_validate(parsed)
        except Exception as e:
            last_error = f"parse/validation error: {e}"
            feedback = f"\n\nPREVIOUS ATTEMPT FAILED: {last_error}\nRetry."
            continue
        errors = validate_graph(plan, set(EXECUTOR_TOOL_REGISTRY))
        if errors:
            last_error = "graph errors: " + "; ".join(errors)
            feedback = f"\n\nPREVIOUS PLAN INVALID: {last_error}\nRetry."
            continue
        return {
            "plan": [s.model_dump() for s in plan.steps],
            "completed": {},
            "failed": [],
        }
    # All retries failed
    return {
        "plan": [],
        "failed": [{"step_id": "planner", "kind": "planner_cap",
                    "detail": f"Could not produce valid plan after 3 attempts. Last: {last_error}"}],
        "status": "error",
    }


## Executor node — invoked one step at a time via `Send`

Receives `{"step": dict, "deps": dict}` from each Send dispatch. Enforces `tc.name == step["tool"]`. Returns `{"completed": {step_id: result}}` which the reducer merges into PlanState.

In [ ]:
executor_llm = llm.bind_tools(EXECUTOR_TOOLS)


EXECUTOR_SYSTEM_PROMPT = """You are an executor. You receive ONE plan step and its
dependency outputs. RUN the step's specified tool with its specified arguments.

You MAY: substitute placeholders in args with concrete values from dependency outputs.
You MUST NOT: use a different tool than specified, or "improve" the args.

Call the specified tool ONCE and return its result.
"""


class ExecutorPayload(TypedDict):
    step: dict
    deps: dict


def executor_node(payload: ExecutorPayload) -> dict:
    """Run ONE step. Returns dict merged into PlanState['completed'] via reducer."""
    step = payload["step"]
    step_id = step["id"]
    declared_tool = step["tool"]
    deps_str = json.dumps(payload.get("deps", {}), default=str)[:4000]

    user_prompt = (
        f"STEP TO EXECUTE:\n"
        f"  id: {step_id}\n"
        f"  description: {step['description']}\n"
        f"  tool: {declared_tool}\n"
        f"  args: {json.dumps(step.get('args', {}))}\n\n"
        f"DEPENDENCY OUTPUTS:\n{deps_str}\n\n"
        f"Call the specified tool ONCE."
    )
    resp = executor_llm.invoke([
        SystemMessage(content=EXECUTOR_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])

    if not resp.tool_calls:
        return {"failed": [{"step_id": step_id, "kind": "no_tool_call",
                            "detail": "executor returned no tool call"}]}
    tc = resp.tool_calls[0]
    if tc["name"] != declared_tool:
        return {"failed": [{"step_id": step_id, "kind": "wrong_tool",
                            "detail": f"executor used {tc['name']} but step.tool was {declared_tool}"}]}

    tool_fn = next((t for t in EXECUTOR_TOOLS if t.name == declared_tool), None)
    if tool_fn is None:
        return {"failed": [{"step_id": step_id, "kind": "unknown_tool",
                            "detail": f"{declared_tool} not in registry"}]}
    try:
        result_str = tool_fn.invoke(tc["args"])
        result = json.loads(result_str)
        return {"completed": {step_id: {"tool": declared_tool, "result": result}}}
    except Exception as e:
        return {"failed": [{"step_id": step_id, "kind": "tool_error",
                            "detail": f"{type(e).__name__}: {e}"}]}


## Dispatcher — the framework value-add

Lab 12's ~70-line `ThreadPoolExecutor` + `threading.Lock` collapses to ~10 lines. Returns `[Send("executor", payload) for s in ready_steps]`; the framework dispatches each as a parallel sub-graph invocation.

In [ ]:
from langgraph.types import Send


def _ready_steps(plan: list[dict], completed: dict, failed_ids: set[str]) -> list[dict]:
    """Steps whose deps are satisfied and that haven't run yet."""
    return [
        s for s in plan
        if (s["id"] not in completed and s["id"] not in failed_ids
            and all(dep in completed for dep in s.get("depends_on", [])))
    ]


def dispatcher_node(state: PlanState) -> list[Send]:
    """Compute ready steps; return one Send per ready step.

    The framework dispatches each Send as a parallel executor invocation;
    results merge into state['completed'] via the _merge_results reducer.
    """
    failed_ids = {f["step_id"] for f in (state.get("failed") or []) if "step_id" in f}
    ready = _ready_steps(state["plan"], state.get("completed", {}), failed_ids)
    if not ready:
        return []
    return [
        Send("executor", {
            "step": s,
            "deps": {dep: state["completed"].get(dep) for dep in s.get("depends_on", [])},
        })
        for s in ready
    ]


## Replanner

Routes to `synthesize` when complete (or cap reached), back to `planner` when failures remain and budget allows. Uses `Command(goto=..., update=...)`.

In [ ]:
from langgraph.types import Command


def replanner_node(state: PlanState) -> Command[Literal["planner", "synthesize"]]:
    """Decide: all done → synthesize; failures + budget → planner; cap → synthesize."""
    plan = state["plan"]
    completed = state.get("completed", {})
    failed = state.get("failed") or []
    replan_count = state.get("replan_count", 0)

    failed_ids = {f["step_id"] for f in failed if "step_id" in f}
    all_handled = all(s["id"] in completed or s["id"] in failed_ids for s in plan)

    # All steps done (no failures) → synthesize
    if all_handled and not failed:
        return Command(goto="synthesize")

    # Replan budget exhausted → synthesize with partial status
    if replan_count >= MAX_REPLANS:
        return Command(goto="synthesize", update={"status": "partial_after_cap"})

    # Failures present, budget remaining → re-plan
    if failed:
        return Command(goto="planner",
                        update={"replan_count": replan_count + 1})

    # Defensive
    return Command(goto="synthesize")


## Synthesizer

Composes the final answer from completed step results. Cites fetched pages inline.

In [ ]:
SYNTHESIZER_SYSTEM_PROMPT = """You are a synthesizer. Given the user's task and a dict
of executed step results, produce a clear answer using ONLY information from the step results.

Rules:
1. Cite fetched pages inline as [1], [2]; list at end as [N] Title — URL.
2. Do not invent claims not supported by step results.
3. If a step failed, surface it.
"""


def synthesize_node(state: PlanState) -> dict:
    """Compose the final answer."""
    task = state["task"]
    completed = state.get("completed", {})
    failed = state.get("failed") or []

    step_lines = []
    for step_id, step_result in completed.items():
        tool_name = step_result.get("tool")
        result = step_result.get("result", {})
        step_lines.append(f"--- {step_id} ({tool_name}) ---\n"
                           f"{json.dumps(result, default=str)[:3000]}")
    failure_lines = ""
    if failed:
        failure_lines = "\n\nFAILED STEPS:\n" + "\n".join(
            f"- {f.get('step_id')}: {f.get('kind')} ({(f.get('detail') or '')[:200]})"
            for f in failed
        )

    user_prompt = (
        f"USER TASK:\n{task}\n\n"
        f"COMPLETED STEPS:\n{chr(10).join(step_lines) or '(none)'}{failure_lines}\n\n"
        f"Compose the final answer."
    )
    resp = llm.invoke([SystemMessage(content=SYNTHESIZER_SYSTEM_PROMPT),
                        HumanMessage(content=user_prompt)])
    return {
        "final_answer": resp.content if isinstance(resp.content, str) else str(resp.content),
        "status": state.get("status") or "ok",
    }


## Wire the graph

Conditional edges use the dispatcher's `list[Send]` return to fan out parallel executors. The `executor → replanner → planner` cycle handles replans.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


def build_plan_execute_graph(checkpointer=None):
    """Build and compile the plan-and-execute graph."""
    b = StateGraph(PlanState)
    b.add_node("planner", planner_node)
    b.add_node("executor", executor_node)
    b.add_node("replanner", replanner_node)
    b.add_node("synthesize", synthesize_node)

    b.add_edge(START, "planner")
    # planner → dispatcher returns list[Send] → fans out to executor
    b.add_conditional_edges("planner", dispatcher_node, ["executor"])
    # executor → replanner (after all parallel Sends complete)
    b.add_edge("executor", "replanner")
    # replanner uses Command to route to "planner" or "synthesize"
    b.add_edge("synthesize", END)

    return b.compile(checkpointer=checkpointer)


graph = build_plan_execute_graph(checkpointer=InMemorySaver())
print("Graph compiled.")
print(f"Nodes: {list(graph.nodes.keys())}")


## End-to-end run

One demonstration run. Watch for the dispatcher firing parallel `Send`s when the plan has steps in a shared `parallel_group`.

In [ ]:
task = (
    "Research recent developments in the Model Context Protocol (MCP) "
    "from 2-3 sources, then summarize the findings."
)
config = {"configurable": {"thread_id": "demo"}, "recursion_limit": 25}

result = graph.invoke({
    "task": task,
    "plan": [],
    "completed": {},
    "failed": [],
    "replan_count": 0,
    "final_answer": "",
    "status": "",
}, config=config)

print("Final answer:")
print("─" * 60)
print(result.get("final_answer", "[no final answer]"))
print()
print(f"Status: {result.get('status')}")
print(f"Replan count: {result.get('replan_count', 0)}")
print(f"Completed steps: {len(result.get('completed', {}))}")
print(f"Failed steps: {len(result.get('failed', []))}")


**Sample output**:

```
Final answer:
────────────────────────────────────────────────────────────
The Model Context Protocol (MCP) is an open standard for connecting
LLMs to external tools [1]. Recent developments include broader
production adoption [2] and expanded server libraries [3]...

[1] Model Context Protocol Overview — https://...
[2] MCP Production Deployments in 2026 — https://...
[3] MCP Server Gallery — https://...

Status: ok
Replan count: 0
Completed steps: 4
Failed steps: 0
```

The planner emitted ~4 steps (one `web_search`, three parallel `fetch_page`s in a parallel_group); the dispatcher fired three concurrent `Send`s for the parallel group; the reducer merged the three executor outputs into `completed`; the replanner saw "all done" and routed to synthesize.

## Synthesis

What the framework absorbed:

- **Manual dispatcher → `Send`**. Lab 12's ~70-line `ThreadPoolExecutor` + `threading.Lock` + completion tracking becomes 10 lines of `[Send(...) for s in ready]`. The biggest single piece of plumbing the framework removes.
- **Per-step locking → reducer**. The `_merge_results` reducer on `completed` handles parallel-update merge without manual locks. The framework guarantees ordering and consistency.
- **Concurrency limits → recursion_limit + provider rate limits**. Lab 12 had `MAX_PARALLEL_EXECUTORS = 3`. LangGraph's `Send` parallelism is bounded by the event loop; rate limits become the practical cap.
- **Replan-as-control-flow → graph cycle**. Lab 12 used a function `plan_and_execute()` with a Python loop. Lab 15 makes it a graph cycle: replanner → planner → dispatcher → executor → replanner. Same behavior; different shape.

What stayed unchanged from Lab 12:

- `PlanStep` / `Plan` Pydantic schemas with `extra="forbid"`.
- `validate_graph()` (Kahn's + 4 checks).
- The five planner-prompt rules.
- Executor's anti-improvement system prompt + `tc.name == step.tool` enforcement.
- The four-cap composition (`MAX_PLAN_STEPS`, `EXECUTOR_MAX_STEPS`, `MAX_REPLANS`).
- Citation preservation in the synthesizer.

What's omitted vs the parent lab: plan-signature dedup (Lab 12 stretch — useful for replanner-thrash protection; not the headline path). Adding it is ~5 lines in the replanner — see Lab 12's solution for the pattern.
